# R1 – BLUEs (látice + columna) y kernels de los dos años (v6)
**Kernel: R.** Corre después de `preparar_datos.py`. Hace lo mismo para **Y21** (2020-21) y **Y22** (2021-22, B5IR).

1. BLUEs con **todas** las parcelas y testigos: `ton.ha ~ GID + (1|ensayo) + (1|ensayo:rep) + (1|ensayo:rep:bloque) + (1|ensayo:columna)`.
   Sin G×T (cada línea está en un solo ensayo). La fila no se incluye: cada bloque ocupa una sola fila (se verifica).
   Parcelas atípicas por Bonferroni-Holm (Bernal-Vasquez et al., 2016). H² de Cullis y sensibilidad con ensayo fijo.
2. Kernels una sola vez con todas las líneas (solo predictores): **A**; **G** (VanRaden); **K** (gaussiano de SNP);
   gaussianos de **DM**, **DM6** y **DM6AUC** (variables estandarizadas).
3. Para DL: **AO** y **GO** = autovectores × √autovalor, exactamente como BGLR (se comprueba contra BGLR).

Salidas por año en `Datos_entrenamiento/`: `BLUEs_<año>.csv`, `y_folds_<año>.csv`, `AO_<año>.csv`, `GO_<año>.csv`, `kernels_<año>.rds`.

In [ ]:
# Paralelo por procesos (mclapply), como en los scripts anteriores: los dos años a la vez y,
# dentro de cada año, los 3 modelos mixtos y las 6 descomposiciones en paralelo.
suppressPackageStartupMessages({ library(lme4); library(emmeans); library(BGLR); library(parallel) })
NUCLEOS <- max(1, detectCores() - 2); cat("Núcleos:", NUCLEOS, "\n")
# Los procesos hijos no escriben en el notebook: se captura su salida y se imprime al final
en_paralelo <- function(años, f) {
  r <- mclapply(años, function(a) { log <- capture.output(v <- f(a)); list(v = v, log = log) },
                mc.cores = min(length(años), NUCLEOS), mc.preschedule = FALSE)
  for (i in seq_along(r)) { if (inherits(r[[i]], "try-error")) stop(r[[i]]); cat(r[[i]]$log, sep = "\n") }
  setNames(lapply(r, `[[`, "v"), años)
}
if (!exists("%||%", envir = baseenv())) `%||%` <- function(x, y) if (is.null(x)) y else x
AÑOS <- c("Y21", "Y22")
D <- "Datos_entrenamiento/"
FORMULA <- "~ GID + (1|ensayo) + (1|ensayo:rep) + (1|ensayo:rep:bloque) + (1|ensayo:columna)"
dir.create("Resultados_R_v6", showWarnings = FALSE)

## 1. BLUEs

In [ ]:
blues <- function(año) {
  cat("\n==========", año, "==========\n")
  y <- read.csv(paste0(D, "rendimiento_parcelas_", año, ".csv"))
  chk <- aggregate(fila ~ ensayo + rep + bloque, data = y, FUN = function(x) length(unique(x)))
  cat("Parcelas:", nrow(y), "| sin rendimiento:", sum(is.na(y$ton.ha)), "| bloques en más de una fila:", sum(chk$fila > 1), "\n")
  d <- subset(y, !is.na(ton.ha)); for (v in c("GID", "ensayo", "rep", "bloque", "columna")) d[[v]] <- factor(d[[v]])
  m0 <- lmer(as.formula(paste("ton.ha", FORMULA)), data = d)
  rs <- residuals(m0) / sigma(m0); ph <- p.adjust(2 * pnorm(-abs(rs)), method = "holm")
  marc <- d[ph < 0.05, ]
  if (nrow(marc) > 0) {
    med <- median(d$ton.ha)
    elim <- do.call(rbind, lapply(split(marc, paste(marc$GID, marc$ensayo)), function(g) {
      todas <- d[d$GID == g$GID[1] & d$ensayo == g$ensayo[1], ]
      if (nrow(g) >= nrow(todas)) g[which.max(abs(g$ton.ha - med)), ] else g }))
    cat("Parcelas atípicas eliminadas:", nrow(elim), "\n"); print(elim[, c("ensayo", "plot", "GID", "rep", "ton.ha")], row.names = FALSE)
    write.csv(elim, paste0("Resultados_R_v6/atipicas_", año, ".csv"), row.names = FALSE)
    d <- droplevels(d[!(d$plot %in% elim$plot), ])
  } else cat("Sin parcelas atípicas\n")
  FH <- "~ 1 + (1|GID) + (1|ensayo) + (1|ensayo:rep) + (1|ensayo:rep:bloque) + (1|ensayo:columna)"
  FF <- "~ GID + ensayo + (1|ensayo:rep) + (1|ensayo:rep:bloque) + (1|ensayo:columna)"
  MOD <- mclapply(c(FORMULA, FH, FF), function(f) lmer(as.formula(paste("ton.ha", f)), data = d),
                  mc.cores = max(1, min(3, floor(NUCLEOS / 2))))
  m <- MOD[[1]]; mh <- MOD[[2]]; mf <- MOD[[3]]
  vc <- as.data.frame(VarCorr(m))[, c("grp", "vcov")]
  cat("Singular:", isSingular(m), "| AIC:", round(AIC(m), 1), "\n"); print(vc)
  bl <- as.data.frame(emmeans(m, "GID", lmer.df = "asymptotic"))[, c("GID", "emmean")]
  names(bl) <- c("GID", "BLUE"); bl$GID <- as.integer(as.character(bl$GID))
  cat("Rango BLUEs:", round(range(bl$BLUE), 2), "| rango parcelas:", round(range(d$ton.ha), 2), "\n")
  vh <- as.data.frame(VarCorr(mh)); vG <- vh$vcov[vh$grp == "GID"]; vE <- vh$vcov[vh$grp == "Residual"]
  rg <- as.data.frame(ranef(mh, condVar = TRUE)); H2 <- 1 - mean(rg$condsd[rg$grpvar == "GID"]^2) / vG
  bf <- as.data.frame(emmeans(mf, "GID", lmer.df = "asymptotic", rg.limit = 50000))[, c("GID", "emmean")]
  bf$GID <- as.integer(as.character(bf$GID))
  rf <- cor(bl$BLUE, bf$emmean[match(bl$GID, bf$GID)])
  cat(sprintf("H2 de Cullis = %.3f (var_G %.4f, var_E %.4f) | correlación con ensayo fijo = %.4f\n", H2, vG, vE, rf))
  write.csv(bl, paste0(D, "BLUEs_", año, ".csv"), row.names = FALSE)
  write.csv(data.frame(año = año, vc, H2 = H2, cor_ensayo_fijo = rf), paste0("Resultados_R_v6/varianzas_BLUE_", año, ".csv"), row.names = FALSE)
  bl
}
BL <- en_paralelo(AÑOS, blues)

## 2. Kernels y AO/GO

In [ ]:
leer_mat <- function(ruta, gids) {
  d <- read.csv(ruta, check.names = FALSE); M <- as.matrix(d[, -1]); rownames(M) <- d$GID
  stopifnot(all(as.character(gids) %in% rownames(M))); M[as.character(gids), , drop = FALSE]
}
gauss <- function(X) { D2 <- as.matrix(dist(X))^2 / ncol(X); exp(-D2 / median(D2[row(D2) > col(D2)])) }
kernels <- function(año) {
  cat("\n==========", año, "==========\n")
  fo <- read.csv(paste0(D, "folds_", año, ".csv")); gids <- fo$GID
  bl <- BL[[año]]; stopifnot(all(gids %in% bl$GID)); y <- bl$BLUE[match(gids, bl$GID)]
  A <- leer_mat(paste0(D, "X_A_", año, ".csv"), gids)[, as.character(gids)]
  M <- leer_mat(paste0(D, "SNP_", año, ".csv"), gids)
  p <- colMeans(M) / 2; W <- sweep(M, 2, 2 * p); G <- tcrossprod(W) / (2 * sum(p * (1 - p)))
  s <- apply(M, 2, sd); s[s == 0] <- 1; K <- gauss(scale(M, center = TRUE, scale = s))
  KER <- list(A = A, G = G, K = K)
  for (img in c("DM", "DM6", "DM6AUC")) {
    X <- leer_mat(paste0(D, "IMG_", img, "_", año, ".csv"), gids)
    pca <- prcomp(X, center = TRUE, scale. = TRUE)               # como en v3: CP que retienen 98 % (mínimo 3)
    k <- max(which(summary(pca)$importance[3, ] >= 0.98)[1], 3)
    KER[[img]] <- gauss(pca$x[, 1:k, drop = FALSE]); cat(img, ":", ncol(X), "variables ->", k, "CP (98 %)\n")
  }
  for (k in names(KER)) dimnames(KER[[k]]) <- list(gids, gids)
  EVD <- mclapply(KER, function(Z) eigen(Z, symmetric = TRUE), mc.cores = max(1, floor(NUCLEOS / 2)))
  for (k in names(KER)) cat(sprintf("  %-6s diag %.3f | autovalor mín %9.5f | autovalores > 1e-7: %d\n", k,
                                    mean(diag(KER[[k]])), min(EVD[[k]]$values), sum(EVD[[k]]$values > 1e-7)))
  for (k in c("A", "G")) {
    e <- EVD[[k]]; keep <- e$values > 1e-7
    O <- e$vectors[, keep] %*% diag(sqrt(e$values[keep])); colnames(O) <- paste0("PC", seq_len(ncol(O)))
    stopifnot(max(abs(tcrossprod(O) - KER[[k]])) < 1e-6)
    write.csv(data.frame(GID = gids, O), paste0(D, k, "O_", año, ".csv"), row.names = FALSE)
  }
  fm <- BGLR(y = y, ETA = list(A = list(K = A, model = "RKHS")), nIter = 10, burnIn = 5, verbose = FALSE, saveAt = paste0("bglr_tmp_AO_", año, "_"))
  AOb <- fm$ETA[[1]]$V %*% diag(sqrt(fm$ETA[[1]]$d))
  AO <- as.matrix(read.csv(paste0(D, "AO_", año, ".csv"), check.names = FALSE)[, -1])
  cc <- sapply(1:min(ncol(AO), ncol(AOb)), function(j) abs(cor(AO[, j], AOb[, j])))
  cat("AO vs BGLR: |r| mediana", round(median(cc), 4), "| columnas con |r| < 0.99:", sum(cc < 0.99), "\n")
  invisible(file.remove(list.files(pattern = paste0("^bglr_tmp_AO_", año, "_"))))
  write.csv(data.frame(GID = gids, trial.no = fo$trial.no, Fold_Prueba = fo$Fold_Prueba, y = y),
            paste0(D, "y_folds_", año, ".csv"), row.names = FALSE)
  saveRDS(list(KERNELS = KER, EVD = EVD, gids = gids, y = y, fold = fo$Fold_Prueba, ensayo = fo$trial.no),
          paste0(D, "kernels_", año, ".rds"))
  cat("Líneas:", length(gids), "| pliegues:", paste(table(fo$Fold_Prueba), collapse = " "), "\n")
  TRUE
}
invisible(en_paralelo(AÑOS, kernels))